# 00 — Data Exploration

This notebook looks at the raw Reddit dataset on S3 to understand what we are working with before any filtering.

### 1. Setup

Import utilities and start Spark.

In [3]:
# Add parent folder to path so we can import utils
import sys
sys.path.insert(0, "..")

# Import helpers from utils.py
from utils import get_spark, load_submissions, load_comments, CRICKET_SUBREDDITS

# Spark functions
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lower, count


### 2. Start Spark session

Build the Spark session using our helper.

In [4]:
# Start Spark
spark = get_spark("00_DataExploration")
spark


:: loading settings :: url = jar:file:/home/ubuntu/spark-3.5.1-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ubuntu/.ivy2/cache
The jars for the packages stored in: /home/ubuntu/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5bdba34f-31f4-4433-aa93-6c86537a0f36;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 455ms :: artifacts dl 10ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evi

### 3. Load submissions

Load the submissions table from S3.

In [3]:
# Load all submissions
subs = load_submissions(spark)


26/04/27 03:15:40 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


### 4. Submissions schema

Print the schema to see all columns.

In [4]:
# Print submissions schema
subs.printSchema()


root
 |-- author: string (nullable = true)
 |-- author_flair_css_class: string (nullable = true)
 |-- author_flair_text: string (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- distinguished: string (nullable = true)
 |-- domain: string (nullable = true)
 |-- edited: double (nullable = true)
 |-- id: string (nullable = true)
 |-- is_self: boolean (nullable = true)
 |-- locked: boolean (nullable = true)
 |-- num_comments: long (nullable = true)
 |-- over_18: boolean (nullable = true)
 |-- quarantine: boolean (nullable = true)
 |-- retrieved_on: long (nullable = true)
 |-- score: long (nullable = true)
 |-- selftext: string (nullable = true)
 |-- stickied: boolean (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- subreddit_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- url: string (nullable = true)
 |-- yyyy: integer (nullable = true)
 |-- mm: integer (nullable = true)



### 5. Load comments

Load the comments table from S3.

In [5]:
# Load all comments
coms = load_comments(spark)


### 6. Comments schema

Print the schema for comments.

In [6]:
# Print comments schema
coms.printSchema()


root
 |-- author: string (nullable = true)
 |-- author_flair_css_class: string (nullable = true)
 |-- author_flair_text: string (nullable = true)
 |-- body: string (nullable = true)
 |-- controversiality: long (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- distinguished: string (nullable = true)
 |-- edited: double (nullable = true)
 |-- gilded: long (nullable = true)
 |-- id: string (nullable = true)
 |-- link_id: string (nullable = true)
 |-- parent_id: string (nullable = true)
 |-- retrieved_on: long (nullable = true)
 |-- score: long (nullable = true)
 |-- stickied: boolean (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- subreddit_id: string (nullable = true)
 |-- yyyy: integer (nullable = true)
 |-- mm: integer (nullable = true)



### 7. Submissions row count

Count total submissions.

In [7]:
# Count submissions
sub_count = subs.count()
print(f"Total submissions: {sub_count:,}")


Total submissions: 567,890,869


### 8. Comments row count

Count total comments.

In [8]:
# Count comments
com_count = coms.count()
print(f"Total comments: {com_count:,}")


Total comments: 3,675,768,958


### 9. Submissions date range

Show which year/month partitions are present.

In [9]:
# Distinct yyyy/mm partitions in submissions
subs.select("yyyy", "mm").distinct().orderBy("yyyy", "mm").show(50)


+----+---+
|yyyy| mm|
+----+---+
|2023|  6|
|2023|  7|
|2023|  8|
|2023|  9|
|2023| 10|
|2023| 11|
|2023| 12|
|2024|  1|
|2024|  2|
|2024|  3|
|2024|  4|
|2024|  5|
|2024|  6|
|2024|  7|
+----+---+



### 10. Comments date range

Same check on comments.

In [10]:
# Distinct yyyy/mm partitions in comments
coms.select("yyyy", "mm").distinct().orderBy("yyyy", "mm").show(50)


+----+---+
|yyyy| mm|
+----+---+
|2023|  6|
|2023|  7|
|2023|  8|
|2023|  9|
|2023| 10|
|2023| 11|
|2023| 12|
|2024|  1|
|2024|  2|
|2024|  3|
|2024|  4|
|2024|  5|
|2024|  6|
|2024|  7|
+----+---+



### 11. Sample submissions

Look at a few rows.

In [11]:
# Show 5 sample submissions
subs.select("subreddit", "title", "score", "num_comments", "author", "yyyy", "mm").show(5, truncate=80)


+---------------+--------------------------------------------------------------------------------+-----+------------+--------------------+----+---+
|      subreddit|                                                                           title|score|num_comments|              author|yyyy| mm|
+---------------+--------------------------------------------------------------------------------+-----+------------+--------------------+----+---+
|kickbasemanager|                                      Spielt Guerreiro bei Bayern von Anfang an?|    1|           0|            nils_koe|2024|  2|
| u_Beastking_17|Married Mississippi Pastor Rickey Scott Sr. Fired Days After Pregnant Mistres...|    1|           0|        Beastking_17|2024|  2|
| jerkbuds012924|                                                                  Dm me for more|    1|           0|Turbulent_Camera6286|2024|  2|
|   Memphissippi|ORGANIZING AN EXCLUSIVE AND ELITE ORGY PARTY!!! Who’s in, contact the admin o...|    1|        

### 12. Sample comments

Look at a few comment rows.

In [12]:
# Show 5 sample comments
coms.select("subreddit", "body", "score", "author", "yyyy", "mm").show(5, truncate=80)


+----------+--------------------------------------------------------------------------------+-----+--------------------+----+---+
| subreddit|                                                                            body|score|              author|yyyy| mm|
+----------+--------------------------------------------------------------------------------+-----+--------------------+----+---+
|   Mercari|Yes I messed that up. I reported a handful of others. One seller says the fis...|    0|Accomplished-Act-126|2023| 12|
|  discgolf|Let's do this! Got a 4 year old daughter that loves to get out on the course ...|    1|       bravo_rambler|2023| 12|
| MWZombies|Man i got over 550.000 and didnt get banned its a money glich but it will be ...|    0|        DR_Waldo9529|2023| 12|
|neoliberal|Johnson and Johnson wants to settle lawsuit showing their talcum powder cause...|   31|     DirkZelenskyy41|2023| 12|
|  sypherpk|Hate it, there are only, like, 6 named locations, and they all feature a vaul.

### 13. Top subreddits overall

Find the 30 subreddits with the most posts.

In [13]:
# Top 30 subreddits by post count
top_subs = (
    subs.groupBy("subreddit")
    .agg(count("*").alias("post_count"))
    .orderBy(col("post_count").desc())
    .limit(30)
)
top_subs.show(30, truncate=False)


+--------------------+----------+
|subreddit           |post_count|
+--------------------+----------+
|dirtyr4r            |3633005   |
|JerkOffChat         |3616759   |
|AskReddit           |2686082   |
|GaySnapchatImages   |2655639   |
|GaySnapchatShare    |2641484   |
|DirtySnapchat       |1986200   |
|PokemonGoRaids      |1341799   |
|slutsofsnapchat     |1334782   |
|PersonalizedGameRecs|1325223   |
|HentaiAndRoleplayy  |1315212   |
|u_GrownBrilliance   |1310625   |
|MonopolyGoTrading   |1234245   |
|relationship_advice |1149649   |
|MassiveCock         |1096306   |
|DirtyChatPals       |1024807   |
|Monopoly_GO         |970421    |
|AutoNewspaper       |946314    |
|GOONED              |917121    |
|ratemycock          |898375    |
|DickPicRequestv2    |863552    |
|teenagers           |862443    |
|gonewild            |850536    |
|Market76            |817392    |
|DisneyNewsfeed      |786909    |
|SluttyConfessions   |732571    |
|Limitlessrp         |731483    |
|jerkbudsHenta

### 14. Cricket subreddits — submissions

Check how many posts each cricket subreddit has.

In [14]:
# Lowercase target list for matching
lower_subs = [s.lower() for s in CRICKET_SUBREDDITS]

# Post counts for our cricket subreddits
cricket_volume = (
    subs.filter(lower(col("subreddit")).isin(lower_subs))
    .groupBy("subreddit")
    .agg(count("*").alias("post_count"))
    .orderBy(col("post_count").desc())
)
cricket_volume.show(50, truncate=False)


+-------------------+----------+
|subreddit          |post_count|
+-------------------+----------+
|Cricket            |65562     |
|CricketShitpost    |47453     |
|ipl                |20812     |
|PakCricket         |9744      |
|RCB                |9416      |
|KolkataKnightRiders|8030      |
|MumbaiIndians      |4405      |
|SunrisersHyderabad |4320      |
|RajasthanRoyals    |2540      |
|delhicapitals      |2522      |
|IndianCricket      |85        |
|Testcricket        |2         |
|kkr                |1         |
+-------------------+----------+



In [16]:
# Comment counts for our cricket subreddits
cricket_com_volume = (
    coms.filter(lower(col("subreddit")).isin(lower_subs))
    .groupBy("subreddit")
    .agg(count("*").alias("comment_count"))
    .orderBy(col("comment_count").desc())
)
cricket_com_volume.show(50, truncate=False)


+-------------------+-------------+
|subreddit          |comment_count|
+-------------------+-------------+
|Cricket            |5238344      |
|CricketShitpost    |572246       |
|ipl                |342387       |
|PakCricket         |176851       |
|RCB                |116592       |
|MumbaiIndians      |72744        |
|SunrisersHyderabad |66807        |
|KolkataKnightRiders|57046        |
|delhicapitals      |42765        |
|RajasthanRoyals    |22633        |
|IndianCricket      |17           |
|AusCricket         |1            |
+-------------------+-------------+



In [ ]:
# Find ALL subreddits that contain cricket-related keywords in their name
from pyspark.sql.functions import lower, col

cricket_keywords = ["cricket", "ipl", "kohli", "dhoni", "rohit", "csk", "rcb", "mumbaiindians", 
                    "kkr", "delhicapitals", "punjabkings", "rajasthanroyals", "sunrisers", 
                    "lucknowsupergiants", "gujarattitans", "hitman", "virat", "mahi", 
                    "thala", "msd", "t20", "odi", "testcricket", "wcc", "cwc", "rr", "srh", "rohitsharma", "IndiaCricket", "CricketShitpost"]

# Build a regex pattern that matches any of these in the subreddit name
pattern = "(?i)(" + "|".join(cricket_keywords) + ")"

# Find all unique subreddits matching the pattern, with post counts
all_cricket_subs = (
    subs.filter(lower(col("subreddit")).rlike(pattern))
    .groupBy("subreddit")
    .agg(count("*").alias("post_count"))
    .orderBy(col("post_count").desc())
)

print(f"Total unique cricket-related subreddits found:")
all_cricket_subs.show(100, truncate=False)

Total unique cricket-related subreddits found:


### 16. Stop Spark

Free up resources.

In [5]:
# Stop Spark session
spark.stop()
